In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_large, resnet50, vgg16
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Change these (or set the corresponding environment variables) if you are
# not running on Kaggle. DATA_ROOT must contain cat_to_name.json.
# PROCESSED_ROOT must contain <split>/<split>.pt as produced by
# preprocessing.ipynb (e.g. train/train.pt), plus idx_to_class.json.
DATA_ROOT = os.environ.get("FLOWER_DATA_ROOT", "/kaggle/input/dataset")
PROCESSED_ROOT = os.environ.get("FLOWER_PROCESSED_ROOT", "/kaggle/input/training-dataset")
CHECKPOINT_ROOT = os.environ.get("FLOWER_CHECKPOINT_ROOT", "/kaggle/working/checkpoints")

In [ ]:
# Optional: explore the Kaggle input directory layout
for dirname, _, _ in os.walk('/kaggle/input'):
    print(dirname)

# Loading the preprocessed dataset
We have processed the original dataset in another ipynb file, the results of that ipynb file are saved in pt files and used here.

In [ ]:
class PreprocessedDataset(Dataset):
    def __init__(self, tensor_file):
        self.data = torch.load(tensor_file)  # Loads output of preprocessing.ipynb

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # (image, label) for train/valid; (image, filename) for test
        return self.data[idx]

# Loading dataset and models (MOBILENT, RESNET,VGG)
The pt files that we are using is on the drive link that we are sharing, please adjust `PROCESSED_ROOT` above if required!

We are also using the original dataset, but we have downloaded it and renamed it as "dataset"; the `cat_to_name.json` file is necessary for the running process.

In [ ]:
train_loader = DataLoader(
    PreprocessedDataset(os.path.join(PROCESSED_ROOT, 'train', 'train.pt')),
    batch_size=32, shuffle=True, num_workers=4  # set num_workers=0 if this hangs locally/on Windows
)
valid_loader = DataLoader(
    PreprocessedDataset(os.path.join(PROCESSED_ROOT, 'valid', 'valid.pt')),
    batch_size=32, shuffle=False, num_workers=4
)
test_loader = DataLoader(
    PreprocessedDataset(os.path.join(PROCESSED_ROOT, 'test', 'test.pt')),
    batch_size=32, shuffle=False, num_workers=4
)

with open(os.path.join(DATA_ROOT, "cat_to_name.json"), "r") as f:
    cat_to_name = json.load(f)
cat_to_name = {int(k): v for k, v in cat_to_name.items()}

# Label-index -> real category-id mapping saved by preprocessing.ipynb.
# ImageFolder assigns indices by alphabetically sorting folder names, so
# this is NOT simply {i: i}. Falls back to an (incorrect) identity mapping
# with a warning if the file hasn't been generated/uploaded yet.
idx_to_class_path = os.path.join(PROCESSED_ROOT, "idx_to_class.json")
if os.path.exists(idx_to_class_path):
    with open(idx_to_class_path) as f:
        idx_to_class = {int(k): v for k, v in json.load(f).items()}
else:
    print("Warning: idx_to_class.json not found - falling back to an identity "
          "mapping, flower-name lookups will likely be wrong. Re-run "
          "preprocessing.ipynb to generate it.")
    idx_to_class = {i: i for i in range(102)}

num_classes = 102

models = {}

# MobileNetV3
model_mobilenet = mobilenet_v3_large(weights='IMAGENET1K_V1')
model_mobilenet.classifier[3] = nn.Linear(model_mobilenet.classifier[3].in_features, num_classes)
models['mobilenet'] = model_mobilenet

# ResNet50
model_resnet = resnet50(weights='IMAGENET1K_V1')
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, num_classes)
models['resnet'] = model_resnet

# VGG16
model_vgg = vgg16(weights='IMAGENET1K_V1')
model_vgg.classifier[6] = nn.Linear(model_vgg.classifier[6].in_features, num_classes)
models['vgg'] = model_vgg

In [ ]:
def train_model(model, train_loader, valid_loader, model_name, epochs=100, lr=0.0001, patience=10):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = StepLR(optimizer, step_size=7, gamma=0.1)
    # Saving setup
    best_val_loss = float('inf')
    patience_counter = 0
    checkpoint_dir = os.path.join(CHECKPOINT_ROOT, model_name)
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pth')
    # Track losses for plotting
    train_losses, val_losses, val_accs = [], [], []

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        running_val_loss = 0.0

        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = running_val_loss / len(valid_loader)
        val_losses.append(avg_val_loss)

        # Calculate metrics
        accuracy  = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
        recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
        f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

        val_accs.append(accuracy * 100)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Acc: {accuracy:.4f}")

        # Step the LR scheduler once per epoch (previously created but never stepped)
        scheduler.step()

        # Model saving (best by val loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
            }
            torch.save(checkpoint, best_checkpoint_path)
            print(f'New best model saved at epoch {epoch+1}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    # Restore the best checkpoint (lowest val loss) instead of returning
    # whatever state the model was left in after the last epoch/early stop.
    best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])

    return model, train_losses, val_losses, val_accs

In [ ]:
def plot_training_curves(train_losses, val_losses, val_accs, model_name):
    """Plot and save training curves for the writeup"""
    plt.figure(figsize=(12, 4))

    # Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', linewidth=2, color='blue')
    plt.plot(val_losses, label='Val Loss', linewidth=2, color='red')
    plt.title(f'{model_name.upper()} - Loss Curves', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Validation accuracy
    plt.subplot(1, 2, 2)
    plt.plot(val_accs, label='Val Accuracy', linewidth=2, color='green')
    plt.title(f'{model_name.upper()} - Val Accuracy', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name}_training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved {model_name}_training_curves.png")

# Main execution with one click training. VGG, RESNET and MobileNet are trained and evaluated with loss curves and accuracy

In [ ]:
# ========================================
# MAIN EXECUTION - ONE CLICK TRAINING
# ========================================
trained_models = {}
best_val_accs = {}

for name in ['mobilenet', 'resnet', 'vgg']:
    print(f"\n{'='*60}")
    print(f"TRAINING {name.upper()}")
    print(f"{'='*60}")

    # 1. TRAIN MODEL
    trained_model, train_losses, val_losses, val_accs = train_model(
        models[name], train_loader, valid_loader, name, epochs=2, lr=0.0001, patience=10
    )
    trained_models[name] = trained_model
    best_val_accs[name] = max(val_accs)

    # 2. PLOT TRAINING CURVES
    plot_training_curves(train_losses, val_losses, val_accs, name)

    # 3. FINAL VALIDATION METRICS (already printed during training)
    print(f"\n✅ {name.upper()} COMPLETE!")
    print(f"Best validation accuracy: {max(val_accs):.2f}%")

print("\n ALL MODELS TRAINED & TESTED!")

# Clear Evaluation

In [ ]:
# NOTE: this reports metrics on the *validation* set, the same data used
# above to pick each model's best checkpoint. The Kaggle `test` split has
# no ground-truth labels (see dataset/sample_submission.csv), so there is
# no held-out labeled data available locally for an unbiased estimate -
# true test-set accuracy can only be seen after submitting predictions
# (generated below) to Kaggle.
final_metrics = {}

for name in ['mobilenet', 'resnet', 'vgg']:
    print(f"\n{name.upper()} Model:")
    print("-" * 40)

    model = trained_models[name]
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    final_metrics[name] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")

# Generate Kaggle Submission
Runs the best-performing model (by validation accuracy) on the unlabeled test set and writes predictions in the `sample_submission.csv` format (`file_name,id`).

In [ ]:
best_model_name = max(final_metrics, key=lambda n: final_metrics[n]['accuracy'])
print(f"Best model by validation accuracy: {best_model_name} "
      f"({final_metrics[best_model_name]['accuracy']:.4f})")

best_model = trained_models[best_model_name]
best_model.eval()

filenames = []
predictions = []

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, preds = torch.max(outputs, 1)

        filenames.extend(names)
        predictions.extend(preds.cpu().numpy().tolist())

submission = pd.DataFrame({'file_name': filenames, 'id': predictions})
submission.to_csv('submission.csv', index=False)
print(f"Saved submission.csv with {len(submission)} predictions using {best_model_name}.")